### 01 — Exploración de datos de Garmin Connect

#### Objetivo

Conocer la estructura y el contenido de los datos que podemos obtener
de Garmin Connect antes de diseñar la extracción del historial completo.

#### Qué haremos

1. Comprobar el acceso a Garmin Connect desde Python.
2. Consultar una pequeña muestra de actividades.
3. Identificar las modalidades de running.
4. Examinar los campos disponibles, sus tipos, unidades y valores ausentes.
5. Documentar qué información necesitamos y qué problemas debemos resolver.

#### Resultado esperado

Una descripción de los datos disponibles y las decisiones necesarias
para construir el proceso de extracción y preparación de Running Analytics.

#### Alcance

Este notebook es exploratorio. Todavía no implementa la descarga completa,
la base de datos ni el análisis de la evolución del entrenamiento.

No guardaremos contraseñas ni archivos de sesión en el notebook
o en el repositorio.

### Datos personales y reproducción del proyecto

Los ejemplos, resultados y gráficos que se muestren corresponden al historial personal de la autora. No representan datos de otra persona ni resultados universales.

Para reproducir el proyecto con tus entrenamientos, utiliza tu propia cuenta de Garmin y genera tus propios archivos con `src/actualizar_running.py`. El acceso se introduce durante la ejecución; no escribas correo, contraseña, códigos de verificación ni tokens en el código, en las salidas o en archivos publicados.

Los datos originales, los preparados y los archivos de sesión se mantienen locales. Las referencias a actividades y cifras en los notebooks de exploración y calidad documentan la extracción estudiada; no se actualizan automáticamente con un historial diferente. Si se publican resultados o gráficos, su contenido visible puede copiarse.


In [ ]:
# Cargamos Pandas para trabajar con tablas.
import pandas as pd

# Importamos Path, una herramienta de Python para trabajar con rutas.
from pathlib import Path

# Cargamos la herramienta de Python para guardar datos en formato JSON.
import json

In [ ]:
# Cargamos Garmin, la herramienta con la que crearemos la conexión.
from garminconnect import Garmin
# Mostramos un mensaje si la importacion funciona
print("Biblioteca de Garmin cargada correctamente")

In [ ]:
# Cargamos la herramienta para introducir la contraseña sin mostrarla.
from getpass import getpass

# Pedimos el correo de la cuenta de Garmin.
email = input("Correo de Garmin: ")

# Pedimos la contraseña y la guardamos temporalmente en memoria.
password = getpass("Contraseña de Garmin: ") 

In [ ]:
# Funcion que solo se ejecuta si Garmin pide un codigo de verificacion
def pedir_codigo():
    return input("Codigo de verificacion de Garmin: ").strip()

# Creamos el cliente que usara nuestras credenciales
cliente =  Garmin(
    email = email,
    password = password,
    prompt_mfa = pedir_codigo
)

# Iniciamos sesion en Garmin Connect
cliente.login()

# Confirmamos que la conexión ha funcionado.
print("Sesión iniciada correctamente")

In [ ]:
# Consultamos las cinco actividades más recientes.
# start=0 indica que empezamos por la primera.
# limit=5 limita la consulta a cinco actividades.
actividades =  cliente.get_activities(start=0,limit=5)

# Mostramos cuántas actividades hemos recibido.
print("Actividades recibidas:", len(actividades))

In [ ]:
# Seleccionamos la primera actividad de la lista.
# Python empieza a contar las posiciones desde 0.
primera_actividad = actividades[0]

# Contamos cuántos campos contiene el diccionario de la primera actividad.
numero_campos = len(primera_actividad)

# Mostramos el resultado.
print("Campos de la primera actividad:", numero_campos)

# Mostramos los campos recibidos
for campo in primera_actividad.keys():
        print(campo)

In [ ]:
# Recorremos las cinco actividades recibidas.
for actividad in actividades:
    # Mostramos el nombre y la informacion sobre el tipo de actividad (nos interesan running solo)
    print(actividad["activityName"], "->", actividad["activityType"])

In [ ]:
# Empezamos sin ninguna carrera seleccionada.
carrera_ejemplo = None

# Revisamos las actividades de la muestra, una por una.
for actividad in actividades:
    # Comprobamos si es de "running"
    if actividad["activityType"]["typeKey"] == "running":
        # Guardamos la primera que encuentre
        carrera_ejemplo = actividad

        # Dejamos de buscar porque tenemos una
        break
    
# Solo consultamos los campos si hemos encontrado una carrera.
if carrera_ejemplo is not None:
    print("Carrera seleccionada:", carrera_ejemplo["activityName"])

    # Contamos los campos de la carrera seleccionada.
    print("Número de campos:", len(carrera_ejemplo))

    # Mostramos los nombres de los campos, uno por línea.
    for campo in carrera_ejemplo:
        print(campo)
else:
    print("No hay actividades de tipo running en esta muestra.")

In [ ]:
# Creamos la tabla usando los datos de la carrera seleccionada.
tabla_ejemplo = pd.DataFrame(
    carrera_ejemplo.items(),
    columns=["campo", "valor"]
)

# Mostramos la tabla con el formato visual del notebook.
# Empezamos por los primeros 20 campos para facilitar la lectura.
display(tabla_ejemplo.head(20))

#### Primera inspección de la respuesta de Garmin

- Consultamos una muestra de cinco actividades recientes.
- La respuesta es una lista de actividades.
- Cada actividad es un diccionario con campos y valores.
- Algunos campos, como activityType, contienen otro diccionario.
- Los campos disponibles pueden variar entre actividades y deportes.
- Seleccionamos una carrera mediante activityType → typeKey = running,
  en lugar de utilizar su posición en la lista.

La tabla campo–valor permite inspeccionar una actividad.
Todavía no representa el historial completo de carreras.

#### Pendiente de revisar

- Qué modalidades de running aparecen en el historial.
- Qué campos están disponibles en el conjunto de carreras.
- Qué significan sus valores y cuáles son sus unidades.
- Qué valores ausentes o problemas de calidad existen.

In [ ]:
# Pedimos hasta 20 actividades usando el filtro de running de Garmin
carreras_muestra = cliente.get_activities(
    start=0,
    limit=20,
    activitytype="running"
)

print("Actividades recibidas:",len(carreras_muestra))

# Revisamos qué modalidades ha incluido el filtro.
for carrera in carreras_muestra:
    print(
        carrera["activityName"],
        "→",
        carrera["activityType"]["typeKey"]
    )

In [ ]:
# Convertimos la lista de carreras en una tabla.
# Cada diccionario se convierte en una fila.
df_muestra = pd.DataFrame(carreras_muestra)

# Consultamos el tamaño de la tabla.
print("Filas y columnas:", df_muestra.shape)

# Mostramos las primeras cinco carreras.
display(df_muestra.head())

In [ ]:
# Creamos un resumen con una fila por cada campo
resumen_campos = pd.DataFrame({
    "tipo": df_muestra.dtypes,
    "con_dato": df_muestra.notna().sum(),
    "vacios": df_muestra.isna().sum()
})

# Vamos a mostrar todas las filas del bloque
with pd.option_context("display.max_rows",None):
    display(resumen_campos)

#### Disponibilidad de campos en la muestra

La muestra contiene 20 actividades de modalidad running y 112 campos.

Tres campos presentan valores ausentes:
- fastestSplit_10000: 14 ausentes.
- workoutId: 14 ausentes.
- fastestSplit_21098: 19 ausentes.

Los otros 109 campos tienen valores presentes en las 20 actividades.
Esto no garantiza que sus valores sean válidos o relevantes.

No eliminamos campos en esta etapa. Las ausencias podrían depender
de las características de cada actividad.

Estas observaciones corresponden únicamente a la muestra consultada,
no al historial completo.

#### Diccionario de datos — Identificación y fechas

| Campo de Garmin | Significado inicial | Utilidad |
|---|---|---|
| activityId | Identificador de la actividad | Identificar carreras y detectar duplicados |
| activityName | Nombre de la actividad | Dar contexto al entrenamiento |
| activityType | Estructura que describe la modalidad | Distinguir modalidades de carrera |
| startTimeLocal | Fecha y hora de inicio local | Situar el entrenamiento en el calendario |
| startTimeGMT | Fecha y hora de inicio en referencia GMT | Mantener una referencia temporal común |

Estas interpretaciones se verificarán con los valores recibidos.
Conservamos los datos originales mientras decidimos las transformaciones.

In [ ]:
# Elegimos los campos para observar
campos_identificacion = [
    "activityId",
    "activityName",
    "startTimeLocal",
    "startTimeGMT"
]

# Mostramos estos campos para las primeras cinco carreras.
display(df_muestra[campos_identificacion].head())

#### Fechas y referencia horaria

- **startTimeLocal:** fecha y hora de inicio según el reloj del lugar donde
  se realizó la actividad. La utilizaremos para agrupar por día, semana y mes.
- **startTimeGMT:** fecha y hora del mismo inicio expresada en la referencia
  GMT, que permite comparar instantes entre distintas zonas horarias.

Conservaremos ambas. La diferencia puede variar con el horario de verano
o la ubicación, por lo que no aplicaremos una diferencia fija de horas.

In [ ]:
# Seleccionamos el nombre, la distancia y las tres medidas de duración.
campos_distancia_tiempo = [
    "activityName",
    "distance",
    "duration",
    "elapsedDuration",
    "movingDuration"
]

# Mostramos una muestra para comparar sus valores.
display(df_muestra[campos_distancia_tiempo].head())

#### Distancia y duración

| Campo | Significado | Unidad |
|---|---|---|
| distance | Distancia registrada | Metros; pendiente de contrastar en Garmin |
| duration | Tiempo del cronómetro, mostrado como «Tiempo total» en Garmin | Segundos |
| elapsedDuration | Tiempo transcurrido, incluidas las pausas del cronómetro | Segundos |
| movingDuration | Tiempo en movimiento calculado por Garmin | Segundos |

Se ha contrastado duration con una actividad en Garmin:
4984,21 segundos corresponden aproximadamente a 1 h 23 min 4 s.

Conservaremos las tres duraciones porque describen aspectos diferentes.
Todavía no hemos decidido cuál utilizar para calcular cada métrica.

In [ ]:
# Seleccionamos distancia, tiempo, velocidad y frecuencia cardiaca.
campos_velocidad_pulso = [
    "distance",
    "duration",
    "averageSpeed",
    "maxSpeed",
    "averageHR",
    "maxHR"
]

# Mostramos las primeras cinco carreras para comparar.
display(df_muestra[campos_velocidad_pulso].head())

#### Velocidad y frecuencia cardiaca

| Campo | Significado | Unidad |
|---|---|---|
| averageSpeed | Velocidad media | Metros por segundo |
| maxSpeed | Velocidad máxima registrada | Metros por segundo |
| averageHR | Frecuencia cardiaca media | Latidos por minuto |
| maxHR | Frecuencia cardiaca máxima registrada | Latidos por minuto |

Para la carrera revisada:
- Frecuencia cardiaca media: 168 latidos/min.
- Frecuencia cardiaca máxima: 187 latidos/min.
- Velocidad máxima: 3,490 m/s, equivalente a 12,6 km/h al redondear.

Los valores coinciden con los mostrados en Garmin Connect.
Conservamos las velocidades originales en m/s; más adelante podremos
crear columnas en km/h y calcular el ritmo en min/km.

In [ ]:
# Seleccionamos los campos relacionados con desnivel y altitud.
campos_elevacion = [
    "activityName",
    "elevationGain",
    "elevationLoss",
    "minElevation",
    "maxElevation",
    "avgElevation"
]

# Observamos sus valores en las primeras cinco carreras.
display(df_muestra[campos_elevacion].head())

#### Desnivel y altitud

| Campo | Significado | Unidad |
|---|---|---|
| elevationGain | Suma de los ascensos del recorrido | Metros |
| elevationLoss | Suma de los descensos del recorrido | Metros |
| minElevation | Altitud mínima del recorrido | Metros |
| maxElevation | Altitud máxima del recorrido | Metros |
| avgElevation | Altitud media del recorrido | Metros |

Las unidades de los cinco campos se han contrastado con Garmin Connect.

En la carrera revisada, el ascenso acumulado es de 96 m y el descenso
acumulado de 94 m; ambos coinciden con Garmin Connect.

El ascenso acumulado no equivale a la diferencia entre la altitud máxima
y la mínima: suma las distintas subidas realizadas durante la actividad.

Conservaremos estos campos para describir el recorrido y contextualizar
las diferencias de ritmo entre entrenamientos.

In [ ]:
# Seleccionamos la cadencia media, máxima y los pasos totales.
campos_cadencia = [
    "activityName",
    "averageRunningCadenceInStepsPerMinute",
    "maxRunningCadenceInStepsPerMinute",
    "steps"
]

# Mostramos las primeras cinco carreras para revisar sus valores.
display(df_muestra[campos_cadencia].head())

#### Cadencia y pasos

| Campo | Significado | Unidad | Validación |
|---|---|---|---|
| averageRunningCadenceInStepsPerMinute | Cadencia media | Pasos/minuto | Contrastada con Garmin |
| maxRunningCadenceInStepsPerMinute | Cadencia máxima | Pasos/minuto | Contrastada con Garmin |
| steps | Número total de pasos de la actividad | Pasos | Pendiente de contrastar |

En la primera carrera, Garmin muestra aproximadamente 169 pasos/min
de media y 177 de máxima, coherentes con los valores recibidos.

In [ ]:
# Seleccionamos los campos de calorías y temperatura.
campos_calorias_temperatura = [
    "activityName",
    "calories",
    "bmrCalories",
    "minTemperature",
    "maxTemperature"
]

# Mostramos las primeras cinco carreras.
display(df_muestra[campos_calorias_temperatura].head())

#### Calorías y temperatura

#### Calorías

| Campo | Significado | Unidad |
|---|---|---|
| calories | Calorías totales estimadas durante la actividad: activas y de reposo | kcal |
| bmrCalories | Calorías de reposo estimadas durante la actividad | kcal |

Valores contrastados con Garmin en la carrera revisada:
- Calorías totales: 864 kcal.
- Calorías de reposo: 98 kcal.
- Calorías activas: 766 kcal.

Las calorías activas se calculan como `calories - bmrCalories`.
No sumaremos ambos campos, porque las calorías totales ya incluyen
las de reposo.

Son estimaciones de Garmin, no mediciones directas del gasto energético.

#### Temperatura

| Campo | Significado | Unidad |
|---|---|---|
| minTemperature | Temperatura mínima registrada durante la actividad | °C |
| maxTemperature | Temperatura máxima registrada durante la actividad | °C |

Valores contrastados con Garmin en la carrera revisada:
- Temperatura mínima: 22 °C.
- Temperatura máxima: 28 °C.

Estos campos no representan la temperatura corporal.
Si la lectura procede del sensor del reloj, puede estar influida
por el calor de la muñeca y la exposición al sol.

Conservaremos ambos campos, pero no los interpretaremos directamente
como temperatura meteorológica. La procedencia del sensor queda
pendiente de confirmar.

In [ ]:
# Seleccionamos los tres campos relacionados con la potencia.
campos_potencia = [
    "activityName",
    "avgPower",
    "maxPower",
    "normPower"
]

# Mostramos sus valores en las primeras cinco carreras.
display(df_muestra[campos_potencia].head())

#### Potencia de carrera

| Campo | Significado | Unidad |
|---|---|---|
| avgPower | Potencia media de la actividad | Vatios (W) |
| maxPower | Potencia máxima registrada | Vatios (W) |
| normPower | Potencia normalizada: resumen que da más peso a los tramos intensos que una media aritmética | Vatios (W) |

Valores contrastados con Garmin en la carrera revisada:
- Potencia media: 248 W.
- Potencia máxima: 399 W.
- Potencia normalizada: 251 W.

La potencia de running es una estimación calculada por Garmin.
Conservaremos los tres campos para explorar la intensidad y variabilidad
de los entrenamientos.

La coincidencia con la interfaz confirma los valores y las unidades;
no verifica por sí sola el algoritmo de cálculo de normPower.

In [ ]:
# Seleccionamos las métricas y etiquetas del efecto del entrenamiento.
campos_efecto = [
    "activityName",
    "aerobicTrainingEffect",
    "anaerobicTrainingEffect",
    "trainingEffectLabel",
    "aerobicTrainingEffectMessage",
    "anaerobicTrainingEffectMessage"
]

# Mostramos cinco carreras.
# Permitimos leer completos los textos solo dentro de este bloque.
with pd.option_context("display.max_colwidth", None):
    display(df_muestra[campos_efecto].head())

#### Efecto del entrenamiento

Estos campos describen el efecto que Garmin estima que ha tenido
cada actividad sobre la capacidad aeróbica y anaeróbica.

| Campo | Significado | Tipo o escala |
|---|---|---|
| aerobicTrainingEffect | Efecto estimado sobre la capacidad de mantener esfuerzos prolongados | Puntuación de 0 a 5 |
| anaerobicTrainingEffect | Efecto estimado sobre la capacidad de realizar esfuerzos cortos y muy intensos | Puntuación de 0 a 5 |
| trainingEffectLabel | Beneficio principal asignado por Garmin | Etiqueta de texto |
| aerobicTrainingEffectMessage | Código descriptivo del efecto aeróbico | Texto |
| anaerobicTrainingEffectMessage | Código descriptivo del efecto anaeróbico | Texto |

#### Interpretación de la escala

| Puntuación | Interpretación general de Garmin |
|---|---|
| 0,0–0,9 | Sin efecto |
| 1,0–1,9 | Efecto menor |
| 2,0–2,9 | Mantenimiento |
| 3,0–3,9 | Mejora |
| 4,0–4,9 | Estímulo elevado de mejora |
| 5,0 | Sobreesfuerzo |

Una puntuación mayor no significa que el entrenamiento sea mejor.
No buscamos maximizar estas puntuaciones en todas las sesiones.

#### Validación con Garmin Connect

En la carrera revisada se han confirmado:
- Efecto aeróbico: 4,5.
- Efecto anaeróbico: 0,1.
- Beneficio principal: umbral, correspondiente a LACTATE_THRESHOLD.

El código NO_ANAEROBIC_BENEFIT_0 indica que Garmin no estima
beneficio anaeróbico. Es un valor válido, no un dato ausente.

#### Uso y limitaciones

Conservaremos las puntuaciones y etiquetas para explorar cómo varían
los estímulos estimados entre entrenamientos.

Son estimaciones de Garmin, no mediciones directas de la mejora física.
Una puntuación de 4,5 no demuestra por sí sola una mejora posterior.
Los mensajes son códigos descriptivos, no variables numéricas:
no interpretaremos sus sufijos como cantidades.

Fuente:
[Training Effect — Garmin](https://www.garmin.com/en-US/garmin-technology/running-science/physiological-measurements/training-effect/)

In [ ]:
# Seleccionamos las métricas de dinámica de carrera.
campos_dinamica = [
    "activityName",
    "avgStrideLength",
    "avgGroundContactTime",
    "avgVerticalOscillation",
    "avgVerticalRatio"
]

# Mostramos las primeras cinco carreras.
display(df_muestra[campos_dinamica].head())

#### Dinámica de carrera

| Campo | Significado | Unidad recibida |
|---|---|---|
| avgStrideLength | Longitud media de zancada | Centímetros |
| avgGroundContactTime | Tiempo medio de contacto del pie con el suelo por paso | Milisegundos |
| avgVerticalOscillation | Desplazamiento vertical medio del torso al correr | Centímetros |
| avgVerticalRatio | Relación entre oscilación vertical y longitud de zancada | Porcentaje |

#### Validación con Garmin Connect

En la carrera revisada se han contrastado:
- Longitud media de zancada: 99,88 cm, aproximadamente 1,00 m.
- Tiempo medio de contacto con el suelo: 265,9 ms.
- Oscilación vertical media: 9,05 cm.
- Relación vertical media: 9,06 %.

La relación vertical expresa la oscilación respecto a la longitud
de zancada. El cociente entre sus promedios puede no coincidir
exactamente con la relación media registrada.

#### Uso y limitaciones

Conservaremos estos campos para explorar cómo cambian con el ritmo
y el tipo de entrenamiento.

No interpretaremos un valor aislado como prueba de buena o mala técnica.
También pueden influir la velocidad, el terreno y las características
personales del corredor.

In [ ]:
# Seleccionamos la duración y el tiempo registrado en las cinco zonas.
campos_zonas_fc = [
    "activityName",
    "duration",
    "hrTimeInZone_1",
    "hrTimeInZone_2",
    "hrTimeInZone_3",
    "hrTimeInZone_4",
    "hrTimeInZone_5"
]

# Mostramos las primeras cinco carreras.
display(df_muestra[campos_zonas_fc].head())

#### Tiempo en zonas de frecuencia cardiaca

| Campo | Significado | Unidad |
|---|---|---|
| hrTimeInZone_1 | Tiempo registrado en la zona 1 | Segundos |
| hrTimeInZone_2 | Tiempo registrado en la zona 2 | Segundos |
| hrTimeInZone_3 | Tiempo registrado en la zona 3 | Segundos |
| hrTimeInZone_4 | Tiempo registrado en la zona 4 | Segundos |
| hrTimeInZone_5 | Tiempo registrado en la zona 5 | Segundos |

Los tiempos se han contrastado con Garmin en la carrera revisada.
Un valor de 0 indica que no se registró tiempo en esa zona; no es un nulo.

En esta actividad, las zonas suman 4984,05 segundos frente a los
4984,21 segundos de duration. Conservamos los valores originales.

### Uso y limitaciones

Utilizaremos estos campos para describir la distribución del tiempo
entre zonas.

Queda pendiente documentar sus límites de pulsaciones y su método
de configuración. No asignaremos categorías de intensidad ni
compararemos periodos sin considerar posibles cambios en las zonas.

In [ ]:
# Seleccionamos la duración y los tiempos de las cinco zonas de potencia.
campos_zonas_potencia = [
    "activityName",
    "duration",
    "powerTimeInZone_1",
    "powerTimeInZone_2",
    "powerTimeInZone_3",
    "powerTimeInZone_4",
    "powerTimeInZone_5"
]

# Mostramos las primeras cinco carreras.
display(df_muestra[campos_zonas_potencia].head())

#### Tiempo en zonas de potencia

Los campos powerTimeInZone_1 a powerTimeInZone_5 representan el tiempo
asignado a cada zona de potencia. Sus valores parecen estar en segundos;
queda pendiente contrastar la unidad con Garmin.

Los límites de las zonas son configurables y deberán documentarse
antes de interpretar la intensidad.

En la primera carrera, las cinco zonas suman 4209,67 segundos,
frente a 4984,21 segundos de duration: hay una diferencia de 774,54 segundos.

Conservamos los valores originales. No asignaremos la diferencia
a ninguna zona ni la trataremos como dato perdido sin investigar su causa.

In [ ]:
# Seleccionamos la distancia total y los campos de mejores tramos.
campos_tramos = [
    "activityName",
    "distance",
    "fastestSplit_1000",
    "fastestSplit_1609",
    "fastestSplit_5000",
    "fastestSplit_10000",
    "fastestSplit_21098"
]

# Mostramos las primeras cinco carreras.
display(df_muestra[campos_tramos].head())

#### Mejores tramos por distancia

Estos campos recogen el mejor tiempo registrado para una distancia
dentro de una actividad. No deben confundirse con las vueltas del reloj
ni con los récords de todo el historial.

| Campo | Distancia del tramo | Unidad del valor |
|---|---|---|
| fastestSplit_1000 | 1 km | Segundos |
| fastestSplit_1609 | Aproximadamente 1 milla | Segundos |
| fastestSplit_5000 | 5 km | Segundos |
| fastestSplit_10000 | 10 km | Segundos |
| fastestSplit_21098 | Aproximadamente media maratón | Segundos |

#### Validación con Garmin

En la carrera revisada se han contrastado:
- Mejor tramo de 1 km: 343,25 segundos, aproximadamente 5:43.
- Mejor tramo de 5 km: 1758,73 segundos, aproximadamente 29:19.

La interpretación de los demás campos queda pendiente de contrastar.

##### Valores ausentes

En la muestra, las actividades que no alcanzan la distancia del tramo
presentan valores ausentes en su campo correspondiente.

Esas ausencias son compatibles con «no aplicable». Revisaremos este
criterio en el historial completo; no asumiremos que explica todos los casos.

No sustituiremos estos valores por cero: cero representaría un tiempo
de realización, no la ausencia de un tramo.

In [ ]:
# Seleccionamos la fecha, el nombre y el valor de VO₂ máximo.
campos_vo2 = [
    "startTimeLocal",
    "activityName",
    "vO2MaxValue"
]

# Mostramos diez carreras para observar si el valor cambia.
display(df_muestra[campos_vo2].head(10))

#### VO₂ máximo — inspección inicial

El campo vO2MaxValue contiene el valor 49,0 en las diez carreras revisadas.

No observamos variación en esta muestra. Esto no demuestra que la
condición física permaneciera exactamente igual ni que Garmin haya
generado una estimación nueva en cada carrera.

Queda pendiente confirmar su significado, unidad y condiciones
de actualización antes de utilizarlo en el análisis.

In [ ]:
# Comparamos los dos indicadores de récord en las 20 carreras.
# Cada comparación devuelve True si ambos valores coinciden.
coinciden_indicadores_pr = df_muestra["isPR"] == df_muestra["pr"]

# Comprobamos si coinciden en todas las filas.
print("¿Coinciden en toda la muestra?", coinciden_indicadores_pr.all())

# Mostramos ambos campos junto al nombre de la carrera.
display(df_muestra[["activityName", "isPR", "pr"]])

#### Criterio de selección de campos

La tabla de trabajo conservará la identificación de la actividad,
las fechas, la modalidad, isPR y los bloques deportivos revisados
en este notebook.

Se excluyen los datos de perfil, configuración, contenido multimedia,
buceo y los demás campos no seleccionados.

En la muestra, isPR y pr coinciden en las 20 carreras. Conservaremos
isPR y comprobaremos esa coincidencia en el historial completo.

Aplicaremos una lista explícita de campos que conservar para que
nuevos campos de Garmin no se incorporen automáticamente.

Si surge una necesidad posterior, revisaremos la incorporación de
campos adicionales.

In [ ]:
# Guardamos los nombres de las columnas que queremos conservar.
columnas_conservar = [
    # Identificación, fechas, modalidad y récord personal.
    "activityId",
    "activityName",
    "startTimeLocal",
    "startTimeGMT",
    "activityType",
    "isPR",

    # Distancia, duración y velocidad.
    "distance",
    "duration",
    "elapsedDuration",
    "movingDuration",
    "averageSpeed",
    "maxSpeed",

    # Frecuencia cardiaca.
    "averageHR",
    "maxHR",

    # Desnivel y altitud.
    "elevationGain",
    "elevationLoss",
    "minElevation",
    "maxElevation",
    "avgElevation",

    # Cadencia y pasos.
    "averageRunningCadenceInStepsPerMinute",
    "maxRunningCadenceInStepsPerMinute",
    "steps",

    # Calorías y temperatura.
    "calories",
    "bmrCalories",
    "minTemperature",
    "maxTemperature",

    # Potencia.
    "avgPower",
    "maxPower",
    "normPower",

    # Efecto del entrenamiento.
    "aerobicTrainingEffect",
    "anaerobicTrainingEffect",
    "trainingEffectLabel",
    "aerobicTrainingEffectMessage",
    "anaerobicTrainingEffectMessage",

    # Dinámica de carrera.
    "avgStrideLength",
    "avgGroundContactTime",
    "avgVerticalOscillation",
    "avgVerticalRatio",

    # Tiempo en zonas de frecuencia cardiaca.
    "hrTimeInZone_1",
    "hrTimeInZone_2",
    "hrTimeInZone_3",
    "hrTimeInZone_4",
    "hrTimeInZone_5",

    # Tiempo en zonas de potencia.
    "powerTimeInZone_1",
    "powerTimeInZone_2",
    "powerTimeInZone_3",
    "powerTimeInZone_4",
    "powerTimeInZone_5",

    # Mejores tramos.
    "fastestSplit_1000",
    "fastestSplit_1609",
    "fastestSplit_5000",
    "fastestSplit_10000",
    "fastestSplit_21098",

    # VO₂ máximo: conservado, con interpretación pendiente.
    "vO2MaxValue"
]

# Contamos cuántos campos hemos seleccionado.
print("Campos seleccionados:", len(columnas_conservar))

In [ ]:
# Empezamos con una lista vacía para guardar los campos que no encontremos.
columnas_faltantes = []

# Revisamos cada nombre de nuestra selección.
for columna in columnas_conservar:
    # Si no existe een la tabla, la añadimos a la lista de faltantes
    if columna not in df_muestra.columns:
        columnas_faltanes.append(columna)
 
# Mostramos los campos que no se han encontrado.
print("Columnas faltantes:", columnas_faltantes)       

In [ ]:
# Seleccionamos las columnas elegidas y creamos una copia independiente
df_running = df_muestra[columnas_conservar].copy()

# Comprobamos el tamaño de la tabla resultante
print("Filas y columnas:",df_running.shape)


#### Tabla de trabajo de la muestra

Creamos df_running seleccionando 54 campos de df_muestra,
que contenía 20 carreras y 112 columnas.

Antes de seleccionar, comprobamos que todos los nombres elegidos
existían en la tabla.

La tabla resultante contiene 20 filas y 54 columnas.
La selección no modifica df_muestra ni los datos de Garmin.

Todavía trabajamos con una muestra. La disponibilidad y calidad
de estos campos se volverán a comprobar en el historial completo.

In [ ]:
# Mostramos su ubicación completa.
print(carpeta_raw.resolve())

# Subimos desde notebooks a running-analytics y entramos en data/raw.
carpeta_raw = Path("../data/raw")


# Unimos la carpeta con el nombre del archivo.
ruta_muestra = carpeta_raw / "garmin_running_muestra_20.json"

# Creamos el archivo sin sobrescribir uno existente.
with ruta_muestra.open("w", encoding="utf-8") as archivo:
    json.dump(
        carreras_muestra,
        archivo,
        ensure_ascii=False,
        indent=2
    )

# Mostramos la ubicación del archivo guardado.
print("Muestra guardada en:", ruta_muestra.resolve())

In [ ]:
# Abrimos en modo lectura: "r"
with ruta_muestra.open("r",encoding="utf-8") as archivo:
    muestra_guardada = json.load(archivo)

# Contamos las actividades recuperadas del archivo.
print("Actividades recuperadas:", len(muestra_guardada))

# Comparamos el contenido leído con la muestra original.
print("¿Coincide con la muestra original?", muestra_guardada == carreras_muestra)


#### Conservación de la muestra original

Guardamos las 20 actividades consultadas en:
data/raw/garmin_running_muestra_20.json

El formato JSON conserva la estructura original, incluidos diccionarios
y listas internos. El archivo contiene todos los campos recibidos,
antes de seleccionar las 54 columnas de trabajo.

Verificamos el guardado leyendo el archivo:
- Se recuperan 20 actividades.
- El contenido coincide con la muestra original.

La ruta se interpreta desde la carpeta de trabajo del notebook,
que en esta sesión es la raíz de running-analytics.

El archivo contiene datos personales y no se publicará en GitHub.

#### Prueba de paginación

Garmin permite consultar las actividades por grupos o páginas.

Después de consultar las primeras 20 carreras, pediremos el segundo
grupo usando start=20 y limit=20.

Esta prueba sirve para entender cómo recuperar el historial completo.
Todavía no implementa la extracción definitiva.

In [ ]:
# Saltamos las primeras 20 carreras y pedimos 20 mas
segunda_pagina = cliente.get_activities(
    start=20,
    limit=20,
    activitytype="running"
)

# Comprobamos cuantas devuelve
print("Carreras recibidas:",len(segunda_pagina))

In [ ]:
# Recogemos los identificadores de la 1º pagina en una lista
ids_primera_pag = []

for carrera in carreras_muestra:
    ids_primera_pag.append(carrera["activityId"])


# Buscamos los ids que estan en la 2º pagina que ya esten en la 1º pagina
ids_repetidos = []

for carrera in segunda_pagina:
    if carrera["activityId"] in ids_primera_pag:
        ids_repetidos.append(carrera["activityid"])

print("IDs compartidos entre páginas:",ids_repetidos)

In [ ]:
# Unimos las dos listas sin modificar las originales
carreras_acumuladas = carreras_muestra + segunda_pagina

print("Total carreras:",len(carreras_acumuladas))

#### Conclusiones de la exploración inicial

- Hemos comprobado la conexión con Garmin Connect.
- Exploramos una muestra de 20 carreras con 112 campos.
- Seleccionamos 54 campos para la tabla de trabajo.
- Documentamos sus significados, unidades y verificaciones pendientes.
- Guardamos y verificamos la muestra original en JSON.
- Probamos la paginación con un segundo grupo de 20 carreras,
  sin identificadores compartidos con la primera muestra.

#### Siguiente fase

Diseñar un proceso independiente para extraer y guardar el historial
completo de running.

Después revisaremos la calidad del historial completo y comprobaremos
si las decisiones tomadas con la muestra siguen siendo adecuadas.